# Manhwa Studio — Google Colab + Google Drive

Versão independente para renderizar com FFmpeg no ambiente do Colab e ler/gravar arquivos diretamente no Google Drive. A interface web original continua separada.

**Uso autorizado:** processe somente vídeos que você possui ou tem autorização para baixar e editar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/ManhwaStudio')
INPUT = WORK / 'entrada'
OUTPUT = WORK / 'exportados'
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Entrada:', INPUT)
print('Saída:', OUTPUT)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!ffmpeg -version | head -n 2
!nvidia-smi || true

In [ ]:
!git clone --depth 1 --branch google-colab-drive https://github.com/manhwavaz1-lab/manhwa-recreator-studio.git /content/manhwa-recreator-studio
%cd /content/manhwa-recreator-studio
!python -m pip install -q yt-dlp

## Execução em um único botão
Cole o link do vídeo, escolha a moldura no upload e clique uma única vez em **EXECUTAR TUDO E BAIXAR MP4**. O notebook vincula os arquivos, renderiza e inicia o download automaticamente.


In [ ]:
from pathlib import Path
videos = sorted([*INPUT.glob('*.mp4'), *INPUT.glob('*.mov'), *INPUT.glob('*.mkv'), *INPUT.glob('*.webm')])
frames = sorted([*INPUT.glob('*.png'), *INPUT.glob('*.jpg'), *INPUT.glob('*.jpeg')])
print('Vídeos:', [p.name for p in videos])
print('Molduras:', [p.name for p in frames])

# Edite estes caminhos se houver mais de um arquivo:
VIDEO = str(videos[0]) if videos else ''
FRAME = str(frames[0]) if frames else ''
print('VIDEO =', VIDEO, '\\nFRAME =', FRAME)

## 2. Opcional: baixar vídeo do YouTube
Use apenas quando tiver direito/autorização para baixar o conteúdo. Deixe `YOUTUBE_URL` vazio se já colocou o MP4 no Drive.

In [ ]:
YOUTUBE_URL = ''  # cole aqui o link, se necessário
if YOUTUBE_URL.strip():
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'yt_dlp', '-f', 'bv*+ba/b', '--merge-output-format', 'mp4', '-o', str(INPUT / 'youtube_source.%(ext)s'), YOUTUBE_URL], check=True)
    downloaded = sorted(INPUT.glob('youtube_source.*'), key=lambda p:p.stat().st_mtime, reverse=True)
    VIDEO = str(downloaded[0])
print('Vídeo selecionado:', VIDEO or 'Nenhum')

## 3. Interface de renderização
Selecione o vídeo, selecione a moldura, escolha o formato e clique em renderizar. O arquivo também será salvo no Google Drive.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from pathlib import Path
import subprocess, sys, re

OUT = str(OUTPUT / 'manhwa-studio-export.mp4')
VIDEO = ''
FRAME = ''

video_url = widgets.Text(
    value='', placeholder='Cole o link do vídeo com Ctrl+V (YouTube ou Google Drive)',
    description='Vídeo:', layout=widgets.Layout(width='98%'),
    style={'description_width':'initial'}
)
frame_upload = widgets.FileUpload(
    accept='.png,.jpg,.jpeg,.webp', multiple=False, description='Escolher moldura'
)
format_select = widgets.Dropdown(options=['16:9','9:16'], value='16:9', description='Formato:')
encoder_select = widgets.Dropdown(options=['auto','nvenc','qsv','amf','cpu'], value='auto', description='Encoder:')
zoom_slider = widgets.IntSlider(value=100, min=50, max=200, step=5, description='Zoom:')
crop_check = widgets.Checkbox(value=True, description='Recortar para preencher')
run_btn = widgets.Button(description='EXECUTAR TUDO E BAIXAR MP4', button_style='success', icon='play', layout=widgets.Layout(width='98%', height='48px'))
status = widgets.Output()

def drive_id(url):
    for pattern in [r'/file/d/([a-zA-Z0-9_-]+)', r'[?&]id=([a-zA-Z0-9_-]+)']:
        match = re.search(pattern, url)
        if match: return match.group(1)
    return None

def fetch_drive(url, dest):
    import gdown
    fid = drive_id(url)
    result = gdown.download(id=fid, output=str(dest), quiet=False, fuzzy=True) if fid else gdown.download(url, str(dest), quiet=False, fuzzy=True)
    if not result or not dest.exists() or dest.stat().st_size == 0:
        raise RuntimeError('Falha no download do Google Drive. Verifique o compartilhamento por link.')
    return str(dest)

def get_frame_upload():
    val = frame_upload.value
    if not val: return None, None
    if isinstance(val, dict):
        name, item = next(iter(val.items()))
    else:
        item = val[0]
        name = item['name']
    return name, item['content']

def execute_all(_):
    global VIDEO, FRAME
    with status:
        clear_output()
        try:
            url = video_url.value.strip()
            if not url:
                raise ValueError('Cole o link do vídeo no campo acima.')
            if 'youtube.com' in url or 'youtu.be' in url:
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'], check=True)
                template = str(INPUT / 'video_youtube.%(ext)s')
                subprocess.run([sys.executable, '-m', 'yt_dlp', '-f', 'bv*+ba/b', '--merge-output-format', 'mp4', '-o', template, url], check=True)
                candidates = sorted(INPUT.glob('video_youtube.*'), key=lambda p:p.stat().st_mtime, reverse=True)
                if not candidates: raise RuntimeError('O download do vídeo não gerou arquivo.')
                VIDEO = str(candidates[0])
            else:
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
                VIDEO = fetch_drive(url, INPUT / 'video_vinculado.mp4')
            print('Vídeo vinculado:', Path(VIDEO).name)

            name, data = get_frame_upload()
            if data is None:
                raise ValueError('Escolha a moldura no campo de upload.')
            suffix = Path(name).suffix.lower()
            if suffix not in ['.png','.jpg','.jpeg','.webp']:
                raise ValueError('A moldura deve ser PNG, JPG ou WEBP.')
            frame_path = INPUT / ('moldura_enviada' + suffix)
            frame_path.write_bytes(bytes(data))
            FRAME = str(frame_path)
            print('Moldura vinculada:', frame_path.name)

            cmd=[sys.executable, 'colab/renderer.py', '--video', VIDEO, '--frame', FRAME,
                 '--output', OUT, '--format', format_select.value, '--encoder', encoder_select.value,
                 '--fps', '30', '--zoom', str(zoom_slider.value)]
            if crop_check.value: cmd.append('--crop')
            subprocess.run(cmd, check=True)
            print('Renderização concluída. Arquivo salvo no Google Drive:', OUT)
            print('Iniciando download do MP4...')
            files.download(OUT)
        except Exception as e:
            print('Não foi possível concluir:', str(e))

run_btn.on_click(execute_all)
display(widgets.HTML('<h3>Manhwa Studio — execução em um clique</h3><p>Cole o link do vídeo com Ctrl+V, escolha a moldura e aperte uma única vez no botão abaixo. O sistema baixa/vincula os arquivos, renderiza e inicia o download do MP4.</p>'))
display(video_url)
display(frame_upload)
display(format_select, encoder_select, zoom_slider, crop_check)
display(run_btn, status)
